In [ ]:
import math
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Times New Roman', 'SimSun']   # 英文数字优先 Times、中文自动宋体
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 16

In [ ]:
import warnings
warnings.filterwarnings("ignore")

**<font color = blue size=3>读取数据</font>**

In [ ]:
data =  pd.read_excel('d:/两年内是否复发/260930数据处理.xlsx',header=2)

In [ ]:
data

In [ ]:
data.columns

In [ ]:
data = data.drop('Monocyte percentage',axis=1)

In [ ]:
data = data.iloc[1:,:]

In [ ]:
import pandas as pd

missing_summary = pd.DataFrame({
    "Variable": data.columns,
    "Missing_N": data.isnull().sum().values,
    "Missing_Rate": (data.isnull().mean().values * 100).round(2)
})

# 按缺失值数量从高到低排序
missing_summary = missing_summary.sort_values(
    by="Missing_N",
    ascending=False
).reset_index(drop=True)

In [ ]:
missing_summary = missing_summary[
    missing_summary["Missing_N"] > 0
].reset_index(drop=True)

display(missing_summary)

In [ ]:
# 计算每列缺失率
missing_rate = data.isnull().mean()

# 找出缺失率 > 80% 的变量
drop_cols = missing_rate[missing_rate > 0.80].index.tolist()

print("删除变量数量：", len(drop_cols))
print("删除的变量：")
print(drop_cols)

# 删除
data = data.drop(columns=drop_cols)

print("删除后数据维度：", data.shape)

In [ ]:
data.info()

In [ ]:
data = data.iloc[:,1:]

In [ ]:
data = data.iloc[:,:-1]

In [ ]:
data

In [ ]:
for col in data.columns:
    print(col)
    print(data[col].unique())

In [ ]:
data.columns

In [ ]:
float_cols = [
    'BMI', 'Hypertension', 'Diabetes',
    'Age', 'γ-Glutamyltransferase', 'ALP', 'ALT', 'AST', 'Homocysteine',
    'Cystatin', 'CRP', 'Creatine kinase', 'Cl', 'Ca', 'Na', 'K', 'P', 'Mg',
    'TBiA', 'TbiL', 'DBiL', 'IBiL', 'HDL-C', 'LDL-C', 'LPa', 'TG', 'LDH',
    'Uric acid', 'TP', 'Albumin', 'Globin', 'ALB/GLO', 'HBeAB', 'HBsAb',
    'HBsAg', 'PT-INR', 'PT', 'PTA', 'TT', 'APTT', 'Lymphocyte count',
    'Lymphocyte percentage', 'Monocyte count',
    'Basophil count', 'Basophil percentage', 'Eosinophil count',
    'Eosinophil percentage', 'Neutrophil count', 'Neutrophil percentage',
    'N/L', 'P/L', 'M/L', 'MCH', 'MCHC', 'MPV', 'RDW-CV', 'Hct', 'PDW',
    'Thrombocrit', 'Platelet count', 'Urine specific gravity',
    'Urine pathological cast count', 'Urine WBC count', 'Urine RBC count',
    'Urine bacterial count', 'Urine pH', 'Urine epithelial cell count',
    'Pathological grade', 'Depth of invasion', 'Tumor size',
    'CT value of lesions on plain scan', 'CT value of pyelography',
    'Tumor location', 'Tumor number', 'Tumor morphology', 'Calcification',
    'Radiologic necrosis', 'Hydronephrosis'
]

# 转为 float
data[float_cols] = data[float_cols].apply(
    pd.to_numeric,
    errors='coerce'
).astype(float)

# Recurrence 转为 int
data['Recurrence'] = pd.to_numeric(
    data['Recurrence'],
    errors='raise'
).astype(int)

# 查看类型
print(data[float_cols + ['Recurrence']].dtypes)

In [ ]:
data.info()

In [ ]:
data.iloc[:,80:]

In [ ]:
data.iloc[:,80:].columns

**<font color = blue size=3>基线分析</font>**

In [ ]:
df = data.copy()

In [ ]:
continuous_features = [ 'BMI','Age', 'γ-Glutamyltransferase', 'ALP', 'ALT', 'AST', 'Homocysteine',
       'Cystatin', 'CRP', 'Creatine kinase', 'Cl', 'Ca', 'Na', 'K', 'P',
                      'Mg', 'TBiA', 'TbiL', 'DBiL', 'IBiL', 'HDL-C', 'LDL-C', 'LPa', 'TG',
       'LDH', 'Uric acid', 'TP', 'Albumin', 'Globin', 'ALB/GLO', 'HBeAB',
       'HBsAb', 'HBsAg', 'PT-INR', 'PT','PTA', 'TT', 'APTT', 'Lymphocyte count', 'Lymphocyte percentage',
       'Monocyte count', 'Basophil count',
       'Basophil percentage', 'Eosinophil count', 'Eosinophil percentage',
       'Neutrophil count', 'Neutrophil percentage', 'N/L', 'P/L', 'M/L', 'MCH',
       'MCHC', 'MPV', 'RDW-CV','Hct', 'PDW', 'Thrombocrit', 'Platelet count', 'Urine specific gravity',
       'Urine pathological cast count', 'Urine WBC count', 'Urine RBC count',
       'Urine bacterial count', 'Urine pH', 'Urine epithelial cell count','Tumor size',
       'CT value of lesions on plain scan', 'CT value of pyelography',]

In [ ]:
discrete_features = ['Gender', 'Smoking', 'Drinking','Hypertension', 'Diabetes','Pathological grade', 'Depth of invasion',
                    'Tumor location', 'Tumor number', 'Tumor morphology', 'Calcification','Radiologic necrosis', 'Hydronephrosis']

In [ ]:
continuous_data = df[continuous_features]
discrete_data = df[discrete_features]

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import (
    shapiro,
    ttest_ind,
    mannwhitneyu,
    chi2_contingency,
    fisher_exact
)

# =========================================================
# 0. 基本配置
# =========================================================
outcome_col = "Recurrence"
output_path = "D:/两年内是否复发/基线特征分析_Table1.xlsx"

# =========================================================
# 1. 样本数统计（用于表头）
# =========================================================
df = df.dropna(subset=[outcome_col])

n_all = df.shape[0]
n_0 = (df[outcome_col] == 0).sum()
n_1 = (df[outcome_col] == 1).sum()

col_overall = f"总体 (n={n_all})"
col_0 = f"{outcome_col}=0 (n={n_0})"
col_1 = f"{outcome_col}=1 (n={n_1})"

# =========================================================
# 2. P 值格式化
# =========================================================
def format_p_with_star(p):
    if p < 0.001:
        return "<0.001***"
    elif p < 0.01:
        return "<0.01**"
    elif p < 0.05:
        return "<0.05*"
    else:
        return f"{p:.3f}"

# =========================================================
# 3. 连续变量基线分析
# =========================================================
def analyze_continuous(df, feature, group):

    g0 = df[df[group] == 0][feature].dropna()
    g1 = df[df[group] == 1][feature].dropna()
    all_data = df[feature].dropna()

    if all_data.empty or g0.empty or g1.empty:
        return []

    # ---------- 总体描述 ----------
    if len(all_data) >= 3 and shapiro(all_data)[1] > 0.05:
        overall_str = f"{all_data.mean():.1f} ± {all_data.std():.1f}"
    else:
        overall_str = (
            f"{all_data.median():.1f} "
            f"({all_data.quantile(0.25):.1f}, {all_data.quantile(0.75):.1f})"
        )

    # ---------- 分组正态性 ----------
    normal0 = len(g0) >= 3 and shapiro(g0)[1] > 0.05
    normal1 = len(g1) >= 3 and shapiro(g1)[1] > 0.05
    normal = normal0 and normal1

    if normal:
        stat, pval = ttest_ind(g0, g1, equal_var=False)
        g0_str = f"{g0.mean():.1f} ± {g0.std():.1f}"
        g1_str = f"{g1.mean():.1f} ± {g1.std():.1f}"
        stat_str = f"t={stat:.2f}"
    else:
        U, pval = mannwhitneyu(g0, g1, alternative="two-sided")
        n0, n1 = len(g0), len(g1)
        mu = n0 * n1 / 2
        sigma = np.sqrt(n0 * n1 * (n0 + n1 + 1) / 12)
        Z = (U - mu) / sigma

        g0_str = (
            f"{g0.median():.1f} "
            f"({g0.quantile(0.25):.1f}, {g0.quantile(0.75):.1f})"
        )
        g1_str = (
            f"{g1.median():.1f} "
            f"({g1.quantile(0.25):.1f}, {g1.quantile(0.75):.1f})"
        )
        stat_str = f"Z={Z:.2f}"

    return [{
        "指标": feature,
        "类别": "",
        col_overall: overall_str,
        col_0: g0_str,
        col_1: g1_str,
        "统计量": stat_str,
        "P值": format_p_with_star(pval)
    }]

# =========================================================
# 4. 分类变量基线分析
# =========================================================
def analyze_categorical(df, feature, group):

    tab = pd.crosstab(df[feature], df[group])

    if tab.shape == (2, 2) and (tab.values < 5).any():
        _, pval = fisher_exact(tab)
        stat_str = "Fisher exact"
    else:
        chi2, pval, _, _ = chi2_contingency(tab)
        stat_str = f"χ²={chi2:.2f}"

    total0 = tab[0].sum()
    total1 = tab[1].sum()
    total_all = total0 + total1

    res = []
    first_row = True

    for level in tab.index:
        count0 = tab.loc[level, 0]
        count1 = tab.loc[level, 1]
        total_k = count0 + count1

        row = {
            "指标": feature if first_row else "",
            "类别": level,
            col_overall: f"{total_k} ({total_k/total_all*100:.1f}%)",
            col_0: f"{count0} ({count0/total0*100:.1f}%)",
            col_1: f"{count1} ({count1/total1*100:.1f}%)",
            "统计量": stat_str if first_row else "",
            "P值": format_p_with_star(pval) if first_row else ""
        }

        res.append(row)
        first_row = False

    return res

# =========================================================
# 5. 生成基线表
# =========================================================
results = []

for col in continuous_features:
    try:
        results.extend(analyze_continuous(df, col, outcome_col))
    except Exception as e:
        print("Error (continuous):", col, e)

for col in discrete_features:
    try:
        results.extend(analyze_categorical(df, col, outcome_col))
    except Exception as e:
        print("Error (categorical):", col, e)

baseline_df = pd.DataFrame(results)

# =========================================================
# 6. 导出 Excel
# =========================================================
baseline_df.to_excel(output_path, index=False)


In [ ]:
baseline_df

**<font color = blue size=3>划分数据集</font>**

In [ ]:
def data_tran(x):
    if x=='男':
        return 1
    else:
        return 0

In [ ]:
df['Gender'] = df['Gender'].apply(lambda x:data_tran(x))

In [ ]:
def data_tran(x):
    if x=='是':
        return 1
    else:
        return 0

In [ ]:
df['Smoking'] = df['Smoking'].apply(lambda x:data_tran(x))

In [ ]:
df['Drinking'] = df['Drinking'].apply(lambda x:data_tran(x))

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=0,
    stratify=df["Recurrence"]
)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge


train_imp = train_df.copy()
test_imp = test_df.copy()


# ============================================================
# 4. 连续变量：IterativeImputer
#    只在训练集拟合
# ============================================================

continuous_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    sample_posterior=False,
    initial_strategy='median'
)

# fit 只使用训练集
train_imp[continuous_features] = continuous_imputer.fit_transform(
    train_df[continuous_features]
)

# 测试集只 transform
test_imp[continuous_features] = continuous_imputer.transform(
    test_df[continuous_features]
)


# ============================================================
# 5. 离散变量：使用训练集众数填补
# ============================================================

discrete_modes = {}

for col in discrete_features:

    # 只计算训练集众数
    mode_value = train_df[col].mode(dropna=True)[0]

    discrete_modes[col] = mode_value

    # 训练集
    train_imp[col] = train_df[col].fillna(mode_value)

    # 测试集也使用训练集众数
    test_imp[col] = test_df[col].fillna(mode_value)


# ============================================================
# 6. 检查填补后的缺失值
# ============================================================

print("=" * 70)
print("训练集剩余缺失值：")
print(train_imp[continuous_features + discrete_features].isnull().sum().sum())

print("\n测试集剩余缺失值：")
print(test_imp[continuous_features + discrete_features].isnull().sum().sum())


# ============================================================
# 7. 查看每个离散变量使用的众数
# ============================================================

print("\n离散变量训练集众数：")

for col, value in discrete_modes.items():
    print(f"{col}: {value}")


# ============================================================
# 8. 数据维度
# ============================================================

print("\n原训练集：", train_df.shape)
print("填补后训练集：", train_imp.shape)

print("原测试集：", test_df.shape)
print("填补后测试集：", test_imp.shape)

In [ ]:
train_df = train_imp.copy()
test_df = test_imp.copy()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import (
    shapiro,
    ttest_ind,
    mannwhitneyu,
    chi2_contingency,
    fisher_exact
)

# ===============================================
# 1. 标记数据来源
# ===============================================
train_df["Dataset"] = "Train"
test_df["Dataset"] = "Test"

df_all = pd.concat([train_df, test_df])

col_train = f"训练集 (n={train_df.shape[0]})"
col_test = f"测试集 (n={test_df.shape[0]})"

# ===============================================
# 2. P值格式
# ===============================================
def format_p(p):
    if p < 0.001:
        return "<0.001***"
    elif p < 0.01:
        return "<0.01**"
    elif p < 0.05:
        return "<0.05*"
    else:
        return f"{p:.3f}"

# ===============================================
# 3. 连续变量比较（关键：U → Z）
# ===============================================
def compare_continuous(feature):

    g_train = df_all[df_all["Dataset"]=="Train"][feature].dropna()
    g_test = df_all[df_all["Dataset"]=="Test"][feature].dropna()

    if len(g_train)<3 or len(g_test)<3:
        return None

    # 正态性检验
    normal_train = shapiro(g_train)[1] > 0.05
    normal_test = shapiro(g_test)[1] > 0.05

    # =========================
    # 正态分布 → t检验
    # =========================
    if normal_train and normal_test:
        stat, p = ttest_ind(g_train, g_test, equal_var=False)
        stat_str = f"t={stat:.2f}"

        train_str = f"{g_train.mean():.2f} ± {g_train.std():.2f}"
        test_str  = f"{g_test.mean():.2f} ± {g_test.std():.2f}"

    # =========================
    # 非正态 → Mann-Whitney → Z
    # =========================
    else:
        U, p = mannwhitneyu(g_train, g_test, alternative='two-sided')

        n1 = len(g_train)
        n2 = len(g_test)

        # U → Z 转换（标准公式）
        mu = n1 * n2 / 2
        sigma = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

        Z = (U - mu) / sigma

        stat_str = f"Z={Z:.2f}"

        train_str = f"{g_train.median():.2f} ({g_train.quantile(0.25):.2f}, {g_train.quantile(0.75):.2f})"
        test_str  = f"{g_test.median():.2f} ({g_test.quantile(0.25):.2f}, {g_test.quantile(0.75):.2f})"

    return {
        "指标": feature,
        col_train: train_str,
        col_test: test_str,
        "统计量": stat_str,
        "P值": format_p(p)
    }

# ===============================================
# 4. 分类变量比较
# ===============================================
def compare_categorical(feature):

    tab = pd.crosstab(df_all[feature], df_all["Dataset"])

    if tab.shape == (2,2) and (tab.values <5).any():
        _, p = fisher_exact(tab)
        stat_str = "Fisher"
    else:
        chi2, p, _, _ = chi2_contingency(tab)
        stat_str = f"χ²={chi2:.2f}"

    res = []
    first = True

    for level in tab.index:
        train_n = tab.loc[level,"Train"]
        test_n  = tab.loc[level,"Test"]

        train_total = tab["Train"].sum()
        test_total  = tab["Test"].sum()

        row = {
            "指标": feature if first else "",
            "类别": level,
            col_train: f"{train_n} ({train_n/train_total*100:.1f}%)",
            col_test: f"{test_n} ({test_n/test_total*100:.1f}%)",
            "统计量": stat_str if first else "",
            "P值": format_p(p) if first else ""
        }

        res.append(row)
        first = False

    return res

# ===============================================
# 5. 生成结果
# ===============================================
results = []

for col in continuous_features:
    try:
        r = compare_continuous(col)
        if r:
            results.append(r)
    except Exception as e:
        print("Error (continuous):", col, e)

# 分类变量（注意加结局）
discrete_features_all = discrete_features + ['Recurrence']

for col in discrete_features_all:
    try:
        r = compare_categorical(col)
        results.extend(r)
    except Exception as e:
        print("Error (categorical):", col, e)

baseline_df = pd.DataFrame(results)

# ===============================================
# 6. 导出
# ===============================================
output_path = "D:/两年内是否复发/Train_vs_Test_基线比较.xlsx"
baseline_df.to_excel(output_path, index=False)

print("已保存：", output_path)

In [ ]:
baseline_df

In [ ]:
train_df = train_df.iloc[:,:-1]
test_df = test_df.iloc[:,:-1]

In [ ]:
train_df

In [ ]:
train_X,test_X,train_Y,test_Y  = train_df.iloc[:,:-1], test_df.iloc[:,:-1], train_df.iloc[:,-1], test_df.iloc[:,-1]

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression


output_dir = Path(r"D:/两年内是否复发")
output_dir.mkdir(parents=True, exist_ok=True)


# =====================================================
# 1. 数据：使用已经完成插补的 train_df、test_df
# =====================================================
train_Y = train_df["Recurrence"].astype(int).copy()
test_Y = test_df["Recurrence"].astype(int).copy()

train_X = train_df.drop(
    columns=["Recurrence", "Dataset", "Monocyte percentage"],
    errors="ignore"
).copy()

test_X = test_df[train_X.columns].copy()

X = train_X.copy()
y = train_Y.copy()


# =====================================================
# 2. 标准化：保留你原来的处理顺序
# =====================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# =====================================================
# 3. Logistic LASSO 交叉验证
# 注意：必须赋值给 Cs
# =====================================================
Cs = np.logspace(-3, 1, 81)

lasso_cv = LogisticRegressionCV(
    Cs=Cs,
    cv=5,
    penalty="l1",
    solver="liblinear",
    scoring="neg_log_loss",
    max_iter=10000,
    random_state=0,
    refit=True
)

lasso_cv.fit(X_scaled, y)


# =====================================================
# 4. Binomial Deviance 与标准误
# =====================================================
scores = np.asarray(
    list(lasso_cv.scores_.values())[0]
)

# 兼容部分版本的返回维度
if scores.ndim == 3 and scores.shape[-1] == 1:
    scores = scores[..., 0]

mean_deviance = -2 * scores.mean(axis=0)

se_deviance = (
    2 * scores.std(axis=0, ddof=1)
    / np.sqrt(scores.shape[0])
)

lambdas = 1 / Cs
x_axis = -np.log(lambdas)


# =====================================================
# 5. lambda.min 和 lambda.1se
# =====================================================
best_idx = int(np.argmin(mean_deviance))

threshold = (
    mean_deviance[best_idx]
    + se_deviance[best_idx]
)

idx_1se = int(
    np.flatnonzero(mean_deviance <= threshold)[0]
)

best_C = float(Cs[best_idx])
best_lambda = float(lambdas[best_idx])

C_1se = float(Cs[idx_1se])
lambda_1se = float(lambdas[idx_1se])

print("lambda.min =", best_lambda)
print("C.min =", best_C)
print("lambda.1se =", lambda_1se)
print("C.1se =", C_1se)


# =====================================================
# 6. 全部变量的系数路径、非零变量数量
# =====================================================
coefs = []

for C in Cs:

    model = LogisticRegression(
        C=float(C),
        penalty="l1",
        solver="liblinear",
        max_iter=10000,
        random_state=0
    )

    model.fit(X_scaled, y)
    coefs.append(model.coef_.ravel())

coefs = np.asarray(coefs)

num_features = np.sum(
    np.abs(coefs) > 1e-10,
    axis=1
)


# =====================================================
# 7. 最终采用 lambda.min 筛选
# =====================================================
lasso_best = LogisticRegression(
    C=best_C,
    penalty="l1",
    solver="liblinear",
    max_iter=10000,
    random_state=0
)

lasso_best.fit(X_scaled, y)

coef = lasso_best.coef_.ravel()

lasso_features_min = X.columns[
    np.abs(coef) > 1e-10
].tolist()

selected_features = lasso_features_min.copy()

print("\nLASSO selected:", len(selected_features))
print(selected_features)

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coef
})

coef_df = coef_df.loc[
    np.abs(coef_df["Coefficient"]) > 1e-10
]

coef_df = coef_df.sort_values(
    by="Coefficient",
    key=np.abs,
    ascending=False
)

print("\nLASSO coefficients:")
print(coef_df)


# =====================================================
# 8. 保存结果
# =====================================================
coef_df.to_csv(
    output_dir / "lasso_coefficients_lambda_min.csv",
    index=False,
    encoding="utf-8-sig"
)

final_data = train_df[
    ["Recurrence"] + selected_features
].copy()

final_data.to_csv(
    output_dir / "lasso_selected_features_lambda_min.csv",
    index=False,
    encoding="utf-8-sig"
)


# =====================================================
# 9. 画图：Binomial Deviance
# =====================================================
fig, ax1 = plt.subplots(figsize=(7, 5))

ax1.errorbar(
    x_axis,
    mean_deviance,
    yerr=se_deviance,
    fmt="o",
    markersize=4,
    color="red",
    ecolor="lightgray",
    elinewidth=1,
    capsize=2
)

ax1.axvline(
    x=x_axis[best_idx],
    linestyle=":",
    color="black",
    linewidth=1.3,
    label="Lambda min"
)

ax1.axvline(
    x=x_axis[idx_1se],
    linestyle="--",
    color="gray",
    linewidth=1.3,
    label="Lambda 1-SE"
)

ax1.set_xlabel(
    r"$-\log(\lambda)$",
    fontsize=13
)

ax1.set_ylabel(
    "Binomial Deviance",
    fontsize=13
)

ax1.tick_params(
    axis="both",
    labelsize=11
)

ax1.legend(frameon=False)


# 顶部显示非零变量数量
ax2 = ax1.secondary_xaxis("top")

tick_idx = np.linspace(
    0,
    len(x_axis) - 1,
    12,
    dtype=int
)

ax2.set_xticks(x_axis[tick_idx])

ax2.set_xticklabels(
    num_features[tick_idx],
    fontsize=11
)

plt.tight_layout()

for ext in ["png", "pdf"]:
    plt.savefig(
        output_dir / f"lasso_binomial_deviance.{ext}",
        dpi=600,
        bbox_inches="tight"
    )

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# =====================================================
# 1. Feature names
# =====================================================
feature_names = X.columns.to_list()

remove_idx = []

# =====================================================
# 2. Compute Logistic LASSO coefficient path
# λ = 1 / C
# =====================================================
coefs = []

for C in Cs:

    model = LogisticRegression(
        C=C,
        penalty="l1",
        solver="liblinear",
        max_iter=10000,
        random_state=0
    )

    model.fit(X_scaled, y)

    coefs.append(model.coef_.ravel())

coefs = np.array(coefs)

lambdas = 1 / Cs
x_axis = -np.log(lambdas)

# =====================================================
# 3. 用 lambda.min / best 找最终特征
# =====================================================
best_idx = np.argmin(mean_deviance)

final_coef = coefs[best_idx]

active_idx = np.where(np.abs(final_coef) > 1e-10)[0]

active_idx = [
    i for i in active_idx
    if i not in remove_idx
]

# =====================================================
# 4. 去掉左边全 0 的区域
# =====================================================
num_features_path = np.sum(np.abs(coefs) > 1e-10, axis=1)

first_nonzero = np.where(num_features_path > 0)[0][0]

x_plot = x_axis[first_nonzero:]
coefs_plot = coefs[first_nonzero:, :]

# =====================================================
# 5. Plot
# =====================================================
plt.figure(figsize=(12, 6))

# Background: all features
# for i in range(coefs_plot.shape[1]):

#     if i in remove_idx:
#         continue

#     plt.plot(
#         x_plot,
#         coefs_plot[:, i],
#         color="lightgray",
#         linewidth=1,
#         alpha=0.6
#     )

# Highlight: lambda.min selected features
for idx in active_idx:

    plt.plot(
        x_axis,
        coefs[:, idx],
        linewidth=2.5,
        label=feature_names[idx]
    )

# lambda.min
plt.axvline(
    x=x_axis[best_idx],
    linestyle=":",
    color="black",
    linewidth=1.3
)

# zero line
plt.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.xlabel(r"$-\log(\lambda)$", fontsize=14, fontweight="bold")
plt.ylabel("Coefficient", fontsize=14, fontweight="bold")
plt.title("LASSO Coefficient Paths", fontsize=16, fontweight="bold")

plt.legend(
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    ncol=1,
    frameon=False,
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "d:/两年内是否复发/lasso_logistic_path.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "d:/两年内是否复发/lasso_logistic_path.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# =====================================
# lambda.min 对应的模型
# =====================================
lasso_final = LogisticRegression(
    C=best_C,
    penalty="l1",
    solver="liblinear",
    max_iter=10000,
    random_state=0
)

lasso_final.fit(X_scaled, y)

# =====================================
# 提取系数
# =====================================
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso_final.coef_.ravel()
})

# 非零系数
coef_df = coef_df[
    coef_df["Coefficient"] != 0
]

coef_df = coef_df.sort_values(
    by="Coefficient",
    key=np.abs,
    ascending=False
)

print(coef_df)

# =====================================
# 保存
# =====================================
coef_df.to_csv(
    "d:/两年内是否复发/lasso_selected_coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 特征列表
# =====================================
selected_features = coef_df["Feature"].tolist()

print("\nSelected Features:")
print(selected_features)

print("\nNumber of Features:")
print(len(selected_features))

In [ ]:
df.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================
# 1. 确保 coef_df 已经是 Logistic LASSO 的结果
# =====================================================
coef_df = coef_df.copy()

coef_df = coef_df[
    coef_df["Coefficient"] != 0
].copy()

# 按系数排序，方便画横向柱状图
coef_df = coef_df.sort_values(
    by="Coefficient",
    ascending=True
)

# =====================================================
# 2. 根据正负系数设置颜色
# =====================================================
colors = coef_df["Coefficient"].apply(
    lambda x: "#D62728" if x > 0 else "#1F77B4"
)

# =====================================================
# 3. 画图
# =====================================================
plt.figure(figsize=(9, 11), dpi=600)

bars = plt.barh(
    coef_df["Feature"],
    coef_df["Coefficient"],
    color=colors,
    edgecolor="black",
    linewidth=0.6
)

# 0 参考线
plt.axvline(
    0,
    color="black",
    linewidth=1.2
)

# =====================================================
# 4. 添加系数数值
# =====================================================
x_min = coef_df["Coefficient"].min()
x_max = coef_df["Coefficient"].max()
offset = (x_max - x_min) * 0.02

for bar in bars:

    width = bar.get_width()

    plt.text(
        width + offset if width > 0 else width - offset,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.3f}",
        va="center",
        ha="left" if width > 0 else "right",
        fontsize=9
    )

# =====================================================
# 5. 坐标轴和标题
# =====================================================
plt.xlabel(
    "Coefficient",
    fontsize=12
)

plt.ylabel("")

plt.title(
    "Logistic LASSO Coefficients",
    fontsize=14,
    weight="bold",
    pad=12
)

plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

sns.despine()

plt.tight_layout()

# =====================================================
# 6. 保存图片
# =====================================================
plt.savefig(
    "d:/两年内是否复发/logistic_lasso_coefficients.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "d:/两年内是否复发/logistic_lasso_coefficients.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier

# ===============================
# 0. 特征名（关键补充）
# ===============================
feature_names = train_X.columns.to_numpy()

# ===============================
# 1. 随机森林基模型
# ===============================
rf_boruta = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=1,
    max_features=0.5,
    class_weight="balanced",
    random_state=42,
    n_jobs=2
)
# ===============================
# 2. Boruta
# ===============================
boruta = BorutaPy(
    estimator=rf_boruta,
    n_estimators="auto",
    max_iter=100,
    perc=80,
    alpha=0.05,
    two_step=True,
    random_state=42,
    verbose=0
)
boruta.fit(train_X.values, train_Y.values)  # ← 推荐用 values

# ===============================
# 3. Boruta 结果
# ===============================
boruta_mask = boruta.support_
boruta_features = feature_names[boruta_mask]

print(f'Boruta 选中特征数: {len(boruta_features)}')
print(boruta_features)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ===============================
# 1. Boruta importance history
# ===============================
imp_hist = np.array(boruta.importance_history_)
imp_hist = imp_hist[~np.all(np.isnan(imp_hist), axis=1)]

imp_df = pd.DataFrame(
    imp_hist,
    columns=feature_names
)

# ===============================
# 2. 特征状态
# ===============================
confirmed = set(np.array(feature_names)[boruta.support_])
tentative = set(np.array(feature_names)[boruta.support_weak_]) - confirmed
rejected = set(feature_names) - confirmed - tentative

def get_status(f):
    if f in confirmed:
        return "confirmed"
    elif f in tentative:
        return "tentative"
    else:
        return "rejected"

# ===============================
# 3. 作图数据
# ===============================
plot_df = imp_df.melt(
    var_name="Feature",
    value_name="Importance"
)

plot_df["Status"] = plot_df["Feature"].apply(get_status)

# ===============================
# 4. 排序：Rejected → Tentative → Confirmed
# ===============================
median_imp = (
    plot_df
    .groupby("Feature")["Importance"]
    .median()
)

rejected_order = (
    median_imp
    .loc[list(rejected)]
    .sort_values()
    .index
    .tolist()
)

tentative_order = (
    median_imp
    .loc[list(tentative)]
    .sort_values()
    .index
    .tolist()
)

confirmed_order = (
    median_imp
    .loc[list(confirmed)]
    .sort_values()
    .index
    .tolist()
)

feature_order = (
    rejected_order
    + tentative_order
    + confirmed_order
)

# ===============================
# 5. 颜色
# ===============================
color_map = {
    "confirmed": "#18A6A6",
    "rejected":  "#E73565",
    "tentative": "#E8B93F"
}

# ===============================
# 6. 绘图
# ===============================
fig, ax = plt.subplots(figsize=(18, 8), dpi=600)

box_data = [
    plot_df.loc[
        plot_df["Feature"] == feat,
        "Importance"
    ].dropna().values
    for feat in feature_order
]

bp = ax.boxplot(
    box_data,
    positions=np.arange(1, len(feature_order) + 1),
    widths=0.65,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=1.8),
    whiskerprops=dict(color="black", linewidth=1.1),
    capprops=dict(color="black", linewidth=1.1),
    boxprops=dict(edgecolor="black", linewidth=1.0),
    flierprops=dict(
        marker="o",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=3.8,
        linestyle="none"
    )
)

for patch, feat in zip(bp["boxes"], feature_order):
    status = get_status(feat)
    patch.set_facecolor(color_map[status])
    patch.set_alpha(0.95)

# ===============================
# 7. 坐标轴
# ===============================
ax.set_xticks(
    np.arange(1, len(feature_order) + 1)
)

ax.set_xticklabels(
    feature_order,
    rotation=90,
    fontsize=18
)

ax.set_ylabel(
    "Importance",
    fontsize=20,
    fontweight="bold"
)

ax.set_xlabel("")

ax.set_title(
    "Boruta Feature Selection",
    fontsize=25,
    fontweight="bold",
    pad=12
)

# ===============================
# 8. 图例
# ===============================
legend_handles = [
    mpatches.Patch(
        facecolor=color_map["confirmed"],
        edgecolor="black",
        label="Confirmed"
    ),
    mpatches.Patch(
        facecolor=color_map["rejected"],
        edgecolor="black",
        label="Rejected"
    ),
    mpatches.Patch(
        facecolor=color_map["tentative"],
        edgecolor="black",
        label="Tentative"
    )
]

ax.legend(
    handles=legend_handles,
    loc="upper left",
    frameon=True,
    fontsize=18,
    edgecolor="lightgray"
)

# ===============================
# 9. 美化
# ===============================
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_linewidth(1.0)

ax.yaxis.grid(
    True,
    linestyle="-",
    linewidth=0.5,
    alpha=0.18
)

ax.xaxis.grid(False)

ax.tick_params(axis="y", labelsize=15)
ax.tick_params(axis="x", length=0)

plt.tight_layout()

plt.savefig(
    "d:/两年内是否复发/Boruta_feature_importance_boxplot.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "d:/两年内是否复发/Boruta_feature_importance_boxplot.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================
# Logistic LASSO 特征
# =====================================
lasso_features = set(selected_features)

# =====================================
# Boruta Confirmed 特征
# =====================================
boruta_features = set(
    np.array(feature_names)[boruta.support_]
)

# =====================================
# 交集
# =====================================
intersect_features = (
    lasso_features &
    boruta_features
)

# =====================================
# 输出
# =====================================
print(f"LASSO特征数: {len(lasso_features)}")
print(f"Boruta Confirmed特征数: {len(boruta_features)}")
print(f"交集特征数: {len(intersect_features)}")

print("\nLASSO Features:")
print(sorted(lasso_features))

print("\nBoruta Features:")
print(sorted(boruta_features))

print("\nIntersection Features:")
print(sorted(intersect_features))

In [ ]:
df.columns

In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt



# =========================
# 1. Figure（比例优化）
# =========================
fig, ax = plt.subplots(figsize=(6.5, 6), dpi=600)

# =========================
# 2. Venn diagram（高级配色）
# =========================
venn = venn2(
    subsets=(lasso_features, boruta_features),
    set_labels=("LASSO", "Boruta"),
    set_colors=("#6baed6", "#fb6a4a"),  # 更高级蓝红
    alpha=0.6,
    ax=ax
)

# =========================
# 3. 标签（顶部）
# =========================
for label in venn.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight("bold")
        label.set_color("#222222")

# =========================
# 4. 数值（核心强调）
# =========================
for text in venn.subset_labels:
    if text:
        text.set_fontsize(16)
        text.set_fontweight("bold")
        text.set_color("#111111")

# =========================
# 5. 圆形边框（高级感关键）
# =========================
for pid in ("10", "01", "11"):
    patch = venn.get_patch_by_id(pid)
    if patch:
        patch.set_edgecolor("#333333")
        patch.set_linewidth(1.8)

# =========================
# 6. 标题（更紧凑）
# =========================
ax.set_title(
    "Feature Selection Overlap (LASSO vs Boruta)",
    fontsize=14,
    fontweight="bold",
    pad=8
)

# =========================
# 7. 去坐标轴（必须）
# =========================
ax.set_axis_off()

# =========================
# 8. 去多余边距（关键）
# =========================
plt.tight_layout(pad=0.5)

# =========================
# 9. 保存（期刊级）
# =========================
plt.savefig("d:/两年内是否复发/LASSO_Boruta_Venn.svg", dpi=600, bbox_inches="tight")
plt.savefig("d:/两年内是否复发/LASSO_Boruta_Venn.png", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
a = list(intersect_features)

In [ ]:
a.append('Recurrence')

In [ ]:
train_df = train_df[a]

In [ ]:
test_df = test_df[a]

In [ ]:
train_df = train_df[['APTT', 'Depth of invasion', 'Pathological grade', 'Radiologic necrosis', 'Smoking', 'TG', 'Tumor location', 
                     'Urine RBC count','Recurrence']]

In [ ]:
test_df = test_df[['APTT', 'Depth of invasion', 'Pathological grade', 'Radiologic necrosis', 'Smoking', 'TG', 'Tumor location', 
                     'Urine RBC count','Recurrence']]

In [ ]:
data_res  = pd.concat([train_df,test_df],axis=0)

In [ ]:
data_res

In [ ]:
train_df

In [ ]:
X_train,X_test,y_train,y_test = train_df.iloc[:,:-1],test_df.iloc[:,:-1],train_df.iloc[:,-1],test_df.iloc[:,-1]

In [ ]:
data_res.to_excel('d:/两年内是否复发/shujuji.xlsx',index=False)

In [ ]:
data_res.info()

In [ ]:


import inspect
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 1. 设置
# ============================================================

SEED = 180
N_BOOTSTRAP = 1000

# 已有完整的新数据框data时，保持None
# 若从Excel读取，填写实际文件路径，例如：
# DATA_FILE = r"D:\两年内是否复发\shujuji(1).xlsx"

DATA_FILE = 'd:/两年内是否复发/shujuji.xlsx'

OUTPUT_FILE = Path(
    r"D:\两年内是否复发\六模型_性能结果.xlsx"
)

continuous_features = [
    "APTT",
    "TG",
    "Urine RBC count",
]

categorical_features = [
    "Depth of invasion",
    "Pathological grade",
    "Radiologic necrosis",
    "Smoking",
    "Tumor location",
]

# 与此次搜索使用的原始列顺序一致
feature_order = [
    "APTT",
    "Depth of invasion",
    "Pathological grade",
    "Radiologic necrosis",
    "Smoking",
    "TG",
    "Tumor location",
    "Urine RBC count",
]


# ============================================================
# 2. 读取与检查数据
# 最后一列为因变量，保留原始行顺序
# ============================================================

if DATA_FILE is not None:
    data = pd.read_excel(DATA_FILE)

elif "data" not in globals() or not isinstance(data, pd.DataFrame):
    raise ValueError(
        "请先读取新数据为data，或填写DATA_FILE。"
    )

df = data.copy().reset_index(drop=True)

if df.shape[1] < 2:
    raise ValueError(
        "数据至少需要一列预测变量和一列因变量。"
    )

if df.columns.duplicated().any():
    raise ValueError("数据存在重复列名。")

if df.isna().any().any():
    raise ValueError(
        "数据存在缺失值，请使用已经处理好的新数据。"
    )

target_col = df.columns[-1]
actual_features = list(df.columns[:-1])

missing_features = [
    column for column in feature_order
    if column not in actual_features
]

extra_features = [
    column for column in actual_features
    if column not in feature_order
]

if missing_features or extra_features:
    raise ValueError(
        "当前data与本次8变量数据不一致。"
        f"缺少列：{missing_features}；"
        f"额外列：{extra_features}。"
    )

y = pd.to_numeric(
    df[target_col],
    errors="raise",
)

if set(y.unique()) != {0, 1}:
    raise ValueError(
        "最后一列因变量必须编码为0和1。"
    )

y = y.astype(int)

X = (
    df[feature_order]
    .apply(pd.to_numeric, errors="raise")
    .copy()
)

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError(
        "预测变量包含无穷大或非有限值。"
    )

for column in continuous_features:
    X[column] = X[column].astype(float)

# CatBoost原生分类特征保留整数编码
# 不能把所有特征统一转换为float
for column in categorical_features:

    values = X[column].to_numpy(dtype=float)

    if not np.equal(values, np.floor(values)).all():
        raise ValueError(
            f"分类变量{column}包含非整数编码，请检查。"
        )

    X[column] = X[column].astype("int64")


# ============================================================
# 3. 六模型共用同一个8:2分层划分
# ============================================================

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=y,
)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

# 保留DataFrame，以支持按列名预处理及CatBoost类别识别
Xtr = X_train
Xte = X_test


# ============================================================
# 4. 与搜索一致的预处理
#
# LR/SVM：连续变量标准化，分类变量独热编码
# RF/XGB/GBDT：连续变量保留原值，分类变量独热编码
# CatBoost：原生分类变量处理
#
# 所有预处理仅在训练集拟合
# ============================================================

def make_preprocessor(scale_continuous=False):

    encoder_kwargs = {
        "drop": "first",
        "handle_unknown": "ignore",
    }

    # 兼容不同版本的scikit-learn
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        encoder_kwargs["sparse_output"] = False
    else:
        encoder_kwargs["sparse"] = False

    return ColumnTransformer([
        (
            "continuous",
            StandardScaler() if scale_continuous else "passthrough",
            continuous_features,
        ),
        (
            "categorical",
            OneHotEncoder(**encoder_kwargs),
            categorical_features,
        ),
    ])


# ============================================================
# 5. 六模型固定参数
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        C=0.3,
        penalty="l2",
        solver="liblinear",
        class_weight=None,
        max_iter=5000,
        random_state=SEED,
    ),

    "SVM": SVC(
        C=100,
        kernel="rbf",
        gamma=0.03,
        class_weight=None,
        probability=False,
        cache_size=200,
        random_state=SEED,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=4,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=1,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.01,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.6,
        reg_alpha=1,
        reg_lambda=30,
        scale_pos_weight=2.3,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
    ),

    "CatBoost": CatBoostClassifier(
        iterations=250,
        depth=6,
        learning_rate=0.01,
        l2_leaf_reg=3,
        random_strength=1,
        auto_class_weights="Balanced",
        one_hot_max_size=2,
        bagging_temperature=1,
        cat_features=categorical_features,
        loss_function="Logloss",
        random_seed=SEED,
        thread_count=1,
        verbose=False,
        allow_writing_files=False,
    ),

    "GBDT": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.01,
        min_samples_leaf=15,
        subsample=0.8,
        max_features=0.7,
        random_state=SEED,
    ),
}


# ============================================================
# 6. 预测得分与训练集Youden阈值
# SVM使用决策得分，其余模型使用正类概率
# ============================================================

def get_score(model, X_input, model_name):

    if model_name == "SVM":
        return np.asarray(
            model.decision_function(X_input)
        ).ravel()

    return np.asarray(
        model.predict_proba(X_input)
    )[:, 1]


def find_best_threshold(y_true, scores):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
    )

    valid = np.isfinite(thresholds)

    if not valid.any():
        raise ValueError("没有可用的有限阈值。")

    youden = tpr[valid] - fpr[valid]
    best_index = np.argmax(youden)

    return float(thresholds[valid][best_index])


# ============================================================
# 7. 性能指标
# AUPRC使用Average Precision
# ============================================================

def calculate_metrics(y_true, scores, y_pred):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if tp + fn > 0 else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0 else np.nan
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0 else 0.0
    )

    f1 = (
        2 * tp / (2 * tp + fp + fn)
        if 2 * tp + fp + fn > 0 else 0.0
    )

    return {
        "AUROC": roc_auc_score(y_true, scores),
        "AUPRC": average_precision_score(y_true, scores),
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "Recall": sensitivity,
        "F1 Score": f1,
    }


# ============================================================
# 8. 测试集Bootstrap 95%CI
# 固定模型、固定训练集阈值
# 不在测试集重新选择阈值
# CI不包含此前选择种子、参数带来的不确定性
# ============================================================

def bootstrap_ci(
    y_true,
    scores,
    y_pred,
    n_bootstrap=1000,
    seed=180,
):

    y_true = np.asarray(y_true).ravel()
    scores = np.asarray(scores).ravel()
    y_pred = np.asarray(y_pred).ravel()

    rng = np.random.RandomState(seed)

    metric_names = [
        "AUROC",
        "AUPRC",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "Recall",
        "F1 Score",
    ]

    bootstrap_values = {
        name: [] for name in metric_names
    }

    for _ in range(n_bootstrap):

        indices = rng.randint(
            0,
            len(y_true),
            len(y_true),
        )

        boot_y = y_true[indices]

        if np.unique(boot_y).size < 2:
            continue

        values = calculate_metrics(
            boot_y,
            scores[indices],
            y_pred[indices],
        )

        for name, value in values.items():
            if np.isfinite(value):
                bootstrap_values[name].append(value)

    ci_results = {}

    for name, values in bootstrap_values.items():

        if len(values) == 0:
            ci_results[name] = (np.nan, np.nan)

        else:

            lower, upper = np.percentile(
                values,
                [2.5, 97.5],
            )

            ci_results[name] = (
                float(lower),
                float(upper),
            )

    return ci_results


# ============================================================
# 9. 训练与评估
# ============================================================

results = []

best_models = {}
best_thresholds = {}
prediction_results = {}

for model_name, estimator in models.items():

    if model_name == "CatBoost":

        model = Pipeline([
            ("model", estimator),
        ])

    else:

        model = Pipeline([
            (
                "preprocessing",
                make_preprocessor(
                    scale_continuous=(
                        model_name in ["Logistic Regression", "SVM"]
                    )
                ),
            ),
            ("model", estimator),
        ])

    model.fit(X_train, y_train)

    best_models[model_name] = model

    train_score = get_score(
        model,
        X_train,
        model_name,
    )

    test_score = get_score(
        model,
        X_test,
        model_name,
    )

    # 使用训练集表观预测确定阈值，不计算OOF
    threshold = find_best_threshold(
        y_train,
        train_score,
    )

    best_thresholds[model_name] = threshold

    # 使用完整精度的训练集阈值预测测试集
    test_pred = (
        test_score >= threshold
    ).astype(int)

    train_auc = roc_auc_score(
        y_train,
        train_score,
    )

    test_metrics = calculate_metrics(
        y_test,
        test_score,
        test_pred,
    )

    ci = bootstrap_ci(
        y_true=y_test,
        scores=test_score,
        y_pred=test_pred,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
    )

    row = {
        "Model": model_name,
        "Train_AUC": round(train_auc, 4),
        "AUC_Gap": round(
            train_auc - test_metrics["AUROC"],
            4,
        ),
        "Threshold": round(threshold, 6),
    }

    for metric_name, value in test_metrics.items():

        lower, upper = ci[metric_name]

        column_name = (
            "Test_AUC (95%CI)"
            if metric_name == "AUROC"
            else f"{metric_name} (95%CI)"
        )

        row[column_name] = (
            f"{value:.3f} "
            f"({lower:.3f}-{upper:.3f})"
        )

    results.append(row)

    # 保留完整精度结果，供后续绘图使用
    prediction_results[model_name] = pd.DataFrame({
        "True_Label": y_test.to_numpy(),
        "Predicted_Score": test_score,
        "Score_Type": (
            "decision_function"
            if model_name == "SVM"
            else "probability"
        ),
        "Threshold": threshold,
        "Predicted_Label": test_pred,
    })


# ============================================================
# 10. 只显示一张表，保存一个Excel
# ============================================================

column_order = [
    "Model",
    "Train_AUC",
    "Test_AUC (95%CI)",
    "AUC_Gap",
    "Threshold",
    "AUPRC (95%CI)",
    "Accuracy (95%CI)",
    "Sensitivity (95%CI)",
    "Specificity (95%CI)",
    "Precision (95%CI)",
    "Recall (95%CI)",
    "F1 Score (95%CI)",
]

results_df = pd.DataFrame(results)[column_order]

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

results_df.to_excel(
    OUTPUT_FILE,
    index=False,
)

display(results_df)

In [ ]:
best_models

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix


# ============================================================
# 1. 获取模型和训练集阈值
# ============================================================

cat_model = best_models["CatBoost"]
threshold = float(best_thresholds["CatBoost"])


# ============================================================
# 2. 预测：保留DataFrame和类别变量的数据类型
# ============================================================

# 不要使用 np.asarray(X_train, dtype=float)
train_prob = cat_model.predict_proba(X_train)[:, 1]
test_prob = cat_model.predict_proba(X_test)[:, 1]

# 训练集、测试集使用同一个训练集阈值
y_train_pred = (train_prob >= threshold).astype(int)
y_test_pred = (test_prob >= threshold).astype(int)

cm_train = confusion_matrix(
    np.asarray(y_train).ravel(),
    y_train_pred,
    labels=[0, 1]
)

cm_test = confusion_matrix(
    np.asarray(y_test).ravel(),
    y_test_pred,
    labels=[0, 1]
)


# ============================================================
# 3. 注释：人数 + 行百分比
# ============================================================

def make_annotations(cm):
    row_sum = cm.sum(axis=1, keepdims=True)

    percentages = np.divide(
        cm.astype(float),
        row_sum,
        out=np.zeros_like(cm, dtype=float),
        where=row_sum != 0
    )

    annotations = np.empty(cm.shape, dtype=object)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annotations[i, j] = (
                f"{cm[i, j]}\n"
                f"{percentages[i, j] * 100:.1f}%"
            )

    return annotations


# ============================================================
# 4. 绘图
# ============================================================

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "none"

# 对应因变量编码：0=未复发，1=复发
class_labels = ["No recurrence", "Recurrence"]

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    dpi=150
)

for ax, cm, title in zip(
    axes,
    [cm_train, cm_test],
    ["Training Set", "Testing Set"]
):
    sns.heatmap(
        cm,
        annot=make_annotations(cm),
        fmt="",
        cmap="Blues",
        cbar=False,
        linewidths=1,
        linecolor="white",
        square=True,
        xticklabels=class_labels,
        yticklabels=class_labels,
        annot_kws={"fontsize": 16},
        ax=ax
    )

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Predicted Label",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        "True Label",
        fontsize=14,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        labelsize=12,
        length=0
    )

    plt.setp(ax.get_xticklabels(), rotation=0)
    plt.setp(ax.get_yticklabels(), rotation=0)

fig.tight_layout()


# ============================================================
# 5. 保存PNG和SVG，仅显示图形
# ============================================================

output_dir = Path(r"D:\两年内是否复发")
output_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    output_dir / "CatBoost_Confusion_Matrix.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    output_dir / "CatBoost_Confusion_Matrix.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import roc_curve, roc_auc_score


# ======================================================
# 1. 设置
# ======================================================

# 这里只控制Bootstrap抽样，不会改变已有的数据划分
BOOTSTRAP_SEED = 180
N_BOOTSTRAP = 1000

output_dir = Path(r"D:\两年内是否复发")
output_dir.mkdir(parents=True, exist_ok=True)

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["svg.fonttype"] = "none"


# ======================================================
# 2. 六模型顺序与配色
# ======================================================

model_order = [
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "XGBoost",
    "CatBoost",
    "GBDT"
]

colors = {
    "Logistic Regression": "#4DBBD5",
    "SVM": "#8491B4",
    "Random Forest": "#3C5488",
    "XGBoost": "#C44E52",
    "CatBoost": "#00A087",
    "GBDT": "#F39B7F"
}

missing_models = [
    name for name in model_order
    if name not in best_models
]

if missing_models:
    raise ValueError(
        f"best_models中缺少模型：{missing_models}，"
        "请先运行最新的六模型训练代码。"
    )

if not (
    isinstance(X_train, pd.DataFrame)
    and isinstance(X_test, pd.DataFrame)
):
    raise TypeError(
        "X_train和X_test必须是保留列名及类别变量类型的DataFrame。"
        "请重新运行最新的训练代码，再运行本段绘图代码。"
    )


# ======================================================
# 3. 获取预测得分
# 保留输入DataFrame，直接使用已拟合的Pipeline
# ======================================================

def get_roc_score(model, X_input, model_name):
    if model_name == "SVM":
        scores = model.decision_function(X_input)
    else:
        # 明确获取标签1对应的概率列
        classes = np.asarray(model.classes_)
        positive_index = np.flatnonzero(classes == 1)

        if positive_index.size != 1:
            raise ValueError(
                f"{model_name}的类别标签不包含唯一的正类1。"
            )

        scores = model.predict_proba(X_input)[
            :, positive_index[0]
        ]

    # 只转换输出得分，不转换输入特征
    return np.asarray(scores, dtype=float).ravel()


# ======================================================
# 4. 实际AUC + Bootstrap 95%CI
# 固定模型预测，对样本进行重抽样
# ======================================================

def bootstrap_auc_ci(
    y_true,
    scores,
    n_bootstrap=1000,
    seed=180
):
    y_true = np.asarray(y_true).ravel()
    scores = np.asarray(scores, dtype=float).ravel()

    if len(y_true) != len(scores):
        raise ValueError("标签与预测得分的样本数不一致。")

    if not np.array_equal(np.unique(y_true), [0, 1]):
        raise ValueError("标签必须包含0和1两类。")

    if not np.isfinite(scores).all():
        raise ValueError("预测得分包含缺失值或无穷值。")

    auc_value = roc_auc_score(y_true, scores)

    rng = np.random.RandomState(seed)
    bootstrap_aucs = []

    for _ in range(n_bootstrap):
        indices = rng.randint(
            0,
            len(y_true),
            size=len(y_true)
        )

        boot_y = y_true[indices]

        if np.unique(boot_y).size < 2:
            continue

        bootstrap_aucs.append(
            roc_auc_score(
                boot_y,
                scores[indices]
            )
        )

    if not bootstrap_aucs:
        raise ValueError("未获得有效的Bootstrap样本。")

    ci_lower, ci_upper = np.percentile(
        bootstrap_aucs,
        [2.5, 97.5]
    )

    return (
        float(auc_value),
        float(ci_lower),
        float(ci_upper)
    )


# ======================================================
# 5. ROC绘图函数
# ======================================================

def plot_six_model_roc(
    X_input,
    y_true,
    title,
    filename
):
    y_true = np.asarray(y_true).ravel()
    roc_info = {}

    # 先计算，避免预测失败时留下空白图
    for name in model_order:
        scores = get_roc_score(
            best_models[name],
            X_input,
            name
        )

        auc_value, ci_lower, ci_upper = bootstrap_auc_ci(
            y_true,
            scores,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED
        )

        fpr, tpr, _ = roc_curve(
            y_true,
            scores,
            pos_label=1
        )

        roc_info[name] = {
            "FPR": fpr,
            "TPR": tpr,
            "AUC": auc_value,
            "CI_Lower": ci_lower,
            "CI_Upper": ci_upper
        }

    fig, ax = plt.subplots(
        figsize=(7, 6),
        dpi=150
    )

    for name in model_order:
        info = roc_info[name]

        ax.plot(
            info["FPR"],
            info["TPR"],
            lw=2.4,
            color=colors[name],
            label=(
                f"{name} "
                f"(AUC = {info['AUC']:.3f} "
                f"[{info['CI_Lower']:.3f}-"
                f"{info['CI_Upper']:.3f}])"
            )
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="gray",
        lw=1.5,
        label="Random classifier"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_axisbelow(True)
    ax.yaxis.grid(
        True,
        linestyle="--",
        alpha=0.35
    )
    ax.xaxis.grid(False)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.tick_params(
        axis="both",
        labelsize=11
    )

    for tick_label in (
        ax.get_xticklabels() + ax.get_yticklabels()
    ):
        tick_label.set_fontweight("bold")

    ax.legend(
        loc="lower right",
        fontsize=10,
        frameon=False,
        title="AUC [95% CI]",
        title_fontsize=11
    )

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=12
    )

    fig.tight_layout()

    fig.savefig(
        output_dir / f"{filename}.png",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_dir / f"{filename}.svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return roc_info


# ======================================================
# 6. 训练集ROC：完整训练集上的表观表现，不计算OOF
# ======================================================

roc_train_info = plot_six_model_roc(
    X_input=X_train,
    y_true=y_train,
    title="Training ROC Curves",
    filename="Training_ROC_Comparison_6Models"
)


# ======================================================
# 7. 测试集ROC
# CI不包含此前选择划分种子及模型参数的不确定性
# ======================================================

roc_test_info = plot_six_model_roc(
    X_input=X_test,
    y_true=y_test,
    title="Testing ROC Curves",
    filename="Testing_ROC_Comparison_6Models"
)

In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve
)
from sklearn.metrics import brier_score_loss


# ======================================================
# 1. 设置
# ======================================================

# 仅控制校准内部的交叉验证，不重新划分训练集和测试集
CALIBRATION_SEED = 180
CALIBRATION_METHOD = "sigmoid"

N_SPLITS = 5
N_BINS = 5
BIN_STRATEGY = "quantile"

output_dir = Path(r"D:\两年内是否复发")
output_dir.mkdir(parents=True, exist_ok=True)

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["svg.fonttype"] = "none"


# ======================================================
# 2. 六模型顺序、颜色、标记
# ======================================================

model_order = [
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "XGBoost",
    "CatBoost",
    "GBDT"
]

colors = {
    "Logistic Regression": "#4DBBD5",
    "SVM": "#8491B4",
    "Random Forest": "#3C5488",
    "XGBoost": "#C44E52",
    "CatBoost": "#00A087",
    "GBDT": "#F39B7F"
}

markers = {
    "Logistic Regression": "o",
    "SVM": "X",
    "Random Forest": "^",
    "XGBoost": "P",
    "CatBoost": "s",
    "GBDT": "v"
}

missing_models = [
    name for name in model_order
    if name not in best_models
]

if missing_models:
    raise ValueError(
        f"缺少模型：{missing_models}，"
        "请先运行最新的六模型训练代码。"
    )


# ======================================================
# 3. 数据检查
# 保留DataFrame、列名及类别变量类型
# ======================================================

if not (
    isinstance(X_train, pd.DataFrame)
    and isinstance(X_test, pd.DataFrame)
):
    raise TypeError(
        "X_train和X_test必须为DataFrame。"
        "请重新运行最新的训练代码，不要将特征转换成浮点数组。"
    )

X_train_cal = X_train.copy()
X_test_cal = X_test.copy()

y_train_array = np.asarray(y_train).ravel()
y_test_array = np.asarray(y_test).ravel()

if len(X_train_cal) != len(y_train_array):
    raise ValueError("训练集特征与标签的样本数不一致。")

if len(X_test_cal) != len(y_test_array):
    raise ValueError("测试集特征与标签的样本数不一致。")

for set_name, labels in [
    ("训练集", y_train_array),
    ("测试集", y_test_array)
]:
    if not np.array_equal(np.unique(labels), [0, 1]):
        raise ValueError(f"{set_name}标签必须包含0和1两类。")

if np.unique(y_train_array, return_counts=True)[1].min() < N_SPLITS:
    raise ValueError(
        f"训练集中每个类别至少需要{N_SPLITS}个样本才能进行校准。"
    )


# ======================================================
# 4. 仅使用训练集拟合概率校正器
#
# clone保留模型参数和Pipeline结构
# 各折内部重新拟合预处理和基础模型
# ensemble=False：最终基础模型在完整训练集上重新拟合
# ======================================================

calibration_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CALIBRATION_SEED
)

calibrated_models = {}
calibrated_train_prob = {}
calibrated_test_prob = {}

calibration_parameters = inspect.signature(
    CalibratedClassifierCV
).parameters

for name in model_order:
    base_model = clone(best_models[name])

    kwargs = {
        "method": CALIBRATION_METHOD,
        "cv": calibration_cv,
        "ensemble": False,
        "n_jobs": 1
    }

    # 兼容不同版本的scikit-learn
    if "estimator" in calibration_parameters:
        kwargs["estimator"] = base_model
    else:
        kwargs["base_estimator"] = base_model

    calibrated_model = CalibratedClassifierCV(**kwargs)

    # 直接传入DataFrame，保留列名和类别类型
    calibrated_model.fit(
        X_train_cal,
        y_train_array
    )

    positive_index = np.flatnonzero(
        np.asarray(calibrated_model.classes_) == 1
    )[0]

    calibrated_models[name] = calibrated_model

    calibrated_train_prob[name] = (
        calibrated_model.predict_proba(X_train_cal)[
            :, positive_index
        ]
    )

    calibrated_test_prob[name] = (
        calibrated_model.predict_proba(X_test_cal)[
            :, positive_index
        ]
    )


# ======================================================
# 5. 绘制校正后的校准曲线
# ======================================================

def plot_calibrated_curves(
    y_true,
    probability_dict,
    title,
    filename
):
    y_true = np.asarray(y_true).ravel()
    calibration_info = {}

    fig, ax = plt.subplots(
        figsize=(7, 6),
        dpi=150
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        color="gray",
        alpha=0.8,
        label="Ideal calibration"
    )

    for name in model_order:
        prob = np.asarray(
            probability_dict[name],
            dtype=float
        ).ravel()

        # 分箱仅用于绘图，不重新拟合校正器
        frac_pos, mean_pred = calibration_curve(
            y_true,
            prob,
            n_bins=N_BINS,
            strategy=BIN_STRATEGY
        )

        brier = brier_score_loss(
            y_true,
            prob,
            pos_label=1
        )

        calibration_info[name] = {
            "Mean_Predicted_Probability": mean_pred,
            "Observed_Frequency": frac_pos,
            "Brier": float(brier)
        }

        ax.plot(
            mean_pred,
            frac_pos,
            marker=markers[name],
            markersize=6,
            linewidth=2.4,
            color=colors[name],
            label=f"{name} (Brier = {brier:.3f})"
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xlabel(
        "Mean predicted probability",
        fontsize=16,
        fontweight="bold"
    )

    ax.set_ylabel(
        "Observed frequency",
        fontsize=16,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        labelsize=13,
        width=1.2
    )

    ax.set_axisbelow(True)
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.legend(
        loc="lower right",
        fontsize=11,
        frameon=False
    )

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=12
    )

    fig.tight_layout()

    fig.savefig(
        output_dir / f"{filename}.png",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_dir / f"{filename}.svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return calibration_info


# ======================================================
# 6. 训练集：校正后的表观校准曲线
# ======================================================

calibration_train_info = plot_calibrated_curves(
    y_true=y_train_array,
    probability_dict=calibrated_train_prob,
    title="Training Calibration (Platt)",
    filename="Training_Calibration_Corrected_6Models"
)


# ======================================================
# 7. 测试集：应用训练集拟合好的校正模型
# ======================================================

calibration_test_info = plot_calibrated_curves(
    y_true=y_test_array,
    probability_dict=calibrated_test_prob,
    title="Testing Calibration (Platt)",
    filename="Testing_Calibration_Corrected_6Models"
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# ======================================================
# 1. 设置
# ======================================================

output_dir = Path(r"D:\两年内是否复发")
output_dir.mkdir(parents=True, exist_ok=True)

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["svg.fonttype"] = "none"

# DCA使用一系列概率阈值，不使用单个Youden阈值
threshold_grid = np.linspace(0.01, 0.99, 100)

# 仅限制显示范围，不修改实际净获益
Y_LIMITS = (-0.10, 0.55)


# ======================================================
# 2. 六模型顺序与配色
# ======================================================

model_order = [
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "XGBoost",
    "CatBoost",
    "GBDT"
]

colors = {
    "Logistic Regression": "#4DBBD5",
    "SVM": "#8491B4",
    "Random Forest": "#3C5488",
    "XGBoost": "#C44E52",
    "CatBoost": "#00A087",
    "GBDT": "#F39B7F"
}


# ======================================================
# 3. 检查模型与数据
# ======================================================

if "calibrated_models" not in globals():
    raise ValueError(
        "请先运行最新的概率校正代码，生成calibrated_models。"
    )

missing_models = [
    name for name in model_order
    if name not in calibrated_models
]

if missing_models:
    raise ValueError(
        f"calibrated_models中缺少：{missing_models}"
    )

if not (
    isinstance(X_train, pd.DataFrame)
    and isinstance(X_test, pd.DataFrame)
):
    raise TypeError(
        "X_train和X_test必须保留DataFrame格式及类别变量类型。"
        "请先运行最新的训练代码和概率校正代码。"
    )


# ======================================================
# 4. 净获益
# NB = TP/N - FP/N × threshold/(1-threshold)
# ======================================================

def net_benefit(probabilities, y_true, thresholds):
    probabilities = np.asarray(
        probabilities, dtype=float
    ).ravel()

    y_true = np.asarray(y_true).ravel()

    thresholds = np.asarray(
        thresholds, dtype=float
    ).ravel()

    if len(y_true) == 0:
        raise ValueError("数据集不能为空。")

    if len(probabilities) != len(y_true):
        raise ValueError("预测概率与真实标签的长度不一致。")

    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("真实标签必须编码为0和1。")

    if (
        not np.isfinite(probabilities).all()
        or np.any(probabilities < 0)
        or np.any(probabilities > 1)
    ):
        raise ValueError(
            "DCA需要0～1的预测概率，不能直接使用SVM决策得分。"
        )

    if (
        not np.isfinite(thresholds).all()
        or np.any(thresholds <= 0)
        or np.any(thresholds >= 1)
    ):
        raise ValueError("DCA阈值必须大于0且小于1。")

    # 每行对应一个样本，每列对应一个阈值
    predicted_positive = (
        probabilities[:, None] >= thresholds[None, :]
    )

    tp = np.sum(
        predicted_positive & (y_true[:, None] == 1),
        axis=0
    )

    fp = np.sum(
        predicted_positive & (y_true[:, None] == 0),
        axis=0
    )

    n = len(y_true)

    return (
        tp / n
        - fp / n * thresholds / (1 - thresholds)
    )


# ======================================================
# 5. 校正后的DCA绘图函数
# ======================================================

def plot_calibrated_dca(
    X_input,
    y_true,
    title,
    filename
):
    # 保留输入DataFrame，不转换为浮点数组
    if not isinstance(X_input, pd.DataFrame):
        raise TypeError("预测输入必须是DataFrame。")

    y_true = np.asarray(y_true).ravel()

    if len(X_input) != len(y_true):
        raise ValueError("特征与标签的样本数不一致。")

    dca_info = {
        "Threshold": threshold_grid.copy()
    }

    # 先完成预测与计算，再创建图形
    for name in model_order:
        model = calibrated_models[name]

        positive_index = np.flatnonzero(
            np.asarray(model.classes_) == 1
        )

        if positive_index.size != 1:
            raise ValueError(
                f"{name}的类别标签不包含唯一的正类1。"
            )

        # 校正后的SVM也使用predict_proba
        probabilities = model.predict_proba(X_input)[
            :, positive_index[0]
        ]

        dca_info[name] = net_benefit(
            probabilities,
            y_true,
            threshold_grid
        )

    # 当前数据集的实际事件比例
    event_rate = np.mean(y_true == 1)

    dca_info["Treat All"] = (
        event_rate
        - (1 - event_rate)
        * threshold_grid / (1 - threshold_grid)
    )

    dca_info["Treat None"] = np.zeros_like(
        threshold_grid
    )

    fig, ax = plt.subplots(
        figsize=(7, 6),
        dpi=150
    )

    for name in model_order:
        ax.plot(
            threshold_grid,
            dca_info[name],
            linewidth=2.5,
            color=colors[name],
            label=name
        )

    ax.plot(
        threshold_grid,
        dca_info["Treat All"],
        linestyle="--",
        linewidth=1.5,
        color="gray",
        label="Treat All"
    )

    ax.plot(
        threshold_grid,
        dca_info["Treat None"],
        linestyle="--",
        linewidth=1.5,
        color="black",
        label="Treat None"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(*Y_LIMITS)

    ax.set_xlabel(
        "Threshold probability",
        fontsize=16,
        fontweight="bold"
    )

    ax.set_ylabel(
        "Net Benefit",
        fontsize=16,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        labelsize=13,
        width=1.2
    )

    ax.set_axisbelow(True)
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.legend(
        loc="upper right",
        fontsize=11,
        frameon=False
    )

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=12
    )

    fig.tight_layout()

    fig.savefig(
        output_dir / f"{filename}.png",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_dir / f"{filename}.svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return dca_info


# ======================================================
# 6. 训练集：校正后模型的表观DCA，不计算OOF
# ======================================================

dca_train_info = plot_calibrated_dca(
    X_input=X_train,
    y_true=y_train,
    title="Training DCA (Calibrated)",
    filename="Training_DCA_Corrected_6Models"
)


# ======================================================
# 7. 测试集：应用训练集拟合好的校正模型
# ======================================================

dca_test_info = plot_calibrated_dca(
    X_input=X_test,
    y_true=y_test,
    title="Testing DCA (Calibrated)",
    filename="Testing_DCA_Corrected_6Models"
)

In [ ]:
# 如果尚未安装shap，在单独的Jupyter单元格运行：
# %pip install shap

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager

import shap
from catboost import CatBoostClassifier, Pool
from scipy.special import expit


# ======================================================
# 1. 设置
# ======================================================

SEED = 180

EXPLAIN_SET = "test"     # "test"或"train"
TOP_N = 6               # 依赖图、PDP展示前6个重要特征
N_DECISION = 30          # 决策图随机展示30个样本

OUTPUT_DIR = Path(r"D:\两年内是否复发\CatBoost_SHAP")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

available_fonts = {
    font.name for font in font_manager.fontManager.ttflist
}

mpl.rcParams.update({
    "font.family": (
        "Times New Roman"
        if "Times New Roman" in available_fonts
        else "DejaVu Serif"
    ),
    "axes.unicode_minus": False,
    "svg.fonttype": "none",
    "figure.dpi": 150,
    "font.size": 11,
})

if EXPLAIN_SET not in {"train", "test"}:
    raise ValueError('EXPLAIN_SET必须为"train"或"test"。')


# ======================================================
# 2. 获取原始CatBoost模型
# 当前训练代码中的CatBoost没有外部预处理步骤
# ======================================================

source_model = best_models["CatBoost"]

if hasattr(source_model, "steps"):
    if len(source_model.steps) != 1:
        raise ValueError(
            "本代码适用于最新训练代码中的单步骤CatBoost Pipeline。"
            "当前模型含额外预处理，请先确认特征对应关系。"
        )
    cat_model = source_model.steps[-1][1]
else:
    cat_model = source_model

if not isinstance(cat_model, CatBoostClassifier):
    raise TypeError(
        'best_models["CatBoost"]中未找到CatBoostClassifier。'
    )

if not np.array_equal(np.asarray(cat_model.classes_), [0, 1]):
    raise ValueError("本代码要求标签编码为0=未复发、1=复发。")


# ======================================================
# 3. 获取解释数据，保留DataFrame及类别类型
# ======================================================

X_selected = X_test if EXPLAIN_SET == "test" else X_train
set_label = "Testing" if EXPLAIN_SET == "test" else "Training"

if not isinstance(X_selected, pd.DataFrame):
    raise TypeError(
        "X_train、X_test必须为DataFrame，"
        "请先运行最新的训练代码。"
    )

feature_names = list(cat_model.feature_names_)

missing_features = [
    name for name in feature_names
    if name not in X_selected.columns
]

if missing_features:
    raise ValueError(f"解释数据缺少特征：{missing_features}")

# 严格使用模型训练时的列顺序
X_explain = (
    X_selected.loc[:, feature_names]
    .copy()
    .reset_index(drop=True)
)

cat_indices = list(cat_model.get_cat_feature_indices())
cat_names = [feature_names[j] for j in cat_indices]

# 当前新数据的类别变量为整数编码
# 检查后恢复整数类型，避免浮点类别导致CatBoost报错
for name in cat_names:
    values = pd.to_numeric(X_explain[name], errors="raise")

    if (
        values.isna().any()
        or not np.isfinite(values.to_numpy(dtype=float)).all()
        or not np.equal(values, np.floor(values)).all()
    ):
        raise ValueError(f"类别变量{name}必须使用无缺失的整数编码。")

    X_explain[name] = values.astype("int64")

n_samples, n_features = X_explain.shape

if n_samples == 0 or n_features == 0:
    raise ValueError("解释数据不能为空。")

# 此数组仅供SHAP绘图使用，不传入CatBoost预测
plot_values = X_explain.to_numpy(dtype=float)


# ======================================================
# 4. 计算CatBoost原生SHAP
# SHAP贡献及基准值均为log-odds尺度
# ======================================================

pool = Pool(
    data=X_explain,
    cat_features=cat_indices
)

shap_output = np.asarray(
    cat_model.get_feature_importance(
        pool,
        type="ShapValues",
        thread_count=1,
        verbose=False
    )
)

shap_values = shap_output[:, :-1]
base_values = shap_output[:, -1]

raw_output = np.asarray(
    cat_model.predict(
        pool,
        prediction_type="RawFormulaVal"
    )
).ravel()

pred_prob = source_model.predict_proba(X_explain)[:, 1]

# 验证SHAP贡献能够还原原始模型输出
np.testing.assert_allclose(
    base_values + shap_values.sum(axis=1),
    raw_output,
    rtol=1e-6,
    atol=1e-6
)

np.testing.assert_allclose(
    expit(raw_output),
    pred_prob,
    rtol=1e-6,
    atol=1e-6
)

np.testing.assert_allclose(
    base_values,
    base_values[0],
    rtol=1e-6,
    atol=1e-6
)

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=base_values,
    data=plot_values,
    feature_names=feature_names
)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

# 使用独立变量名，避免覆盖训练代码中的feature_order
shap_feature_order = np.argsort(-mean_abs_shap)
top_indices = shap_feature_order[:min(TOP_N, n_features)]

shap_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Mean_abs_SHAP": mean_abs_shap
    })
    .sort_values("Mean_abs_SHAP", ascending=False)
    .reset_index(drop=True)
)


# ======================================================
# 5. 绘图辅助函数
# ======================================================

def save_show(fig, filename):
    stem = OUTPUT_DIR / f"{set_label}_{filename}"

    fig.savefig(
        f"{stem}.png",
        dpi=600,
        bbox_inches="tight"
    )
    fig.savefig(
        f"{stem}.svg",
        bbox_inches="tight"
    )
    fig.savefig(
        f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.25
    )
    ax.set_axisbelow(True)


# ======================================================
# 6. 摘要图：平均绝对SHAP值
# ======================================================

plt.figure()

shap.plots.bar(
    shap_explanation,
    max_display=n_features,
    show_data=False,
    show=False
)

fig = plt.gcf()
fig.set_size_inches(
    9,
    max(5, 0.40 * n_features + 1.5)
)

ax = plt.gca()
ax.set_xlabel("Mean absolute SHAP value (log-odds)")
ax.set_title(
    f"CatBoost SHAP Importance - {set_label} Set",
    pad=16
)

fig.tight_layout()
save_show(fig, "CatBoost_SHAP_Summary_Bar")


# ======================================================
# 7. 蜂群图
# 类别变量颜色表示编码值，并不代表连续数量关系
# ======================================================

rng_state = np.random.get_state()
np.random.seed(SEED)

try:
    plt.figure()

    shap.plots.beeswarm(
        shap_explanation,
        max_display=n_features,
        plot_size=(10, max(5, 0.40 * n_features + 1.5)),
        show=False
    )
finally:
    np.random.set_state(rng_state)

fig = plt.gcf()
ax = plt.gca()

ax.set_xlabel("SHAP value (impact on log-odds)")
ax.set_title(
    f"CatBoost SHAP Beeswarm - {set_label} Set",
    pad=16
)

fig.tight_layout()
save_show(fig, "CatBoost_SHAP_Beeswarm")


# ======================================================
# 8. SHAP依赖图
# 类别变量只展示实际类别编码
# ======================================================

ncols = min(3, len(top_indices))
nrows = int(np.ceil(len(top_indices) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5 * ncols, 4 * nrows),
    squeeze=False
)

for ax, j in zip(axes.ravel(), top_indices):
    name = feature_names[j]

    ax.scatter(
        plot_values[:, j],
        shap_values[:, j],
        color="#008F7A",
        s=25,
        alpha=0.70,
        edgecolors="none"
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1
    )

    if name in cat_names:
        ax.set_xticks(np.sort(X_explain[name].unique()))

    ax.set_title(name, fontsize=13)
    ax.set_xlabel(name)
    ax.set_ylabel("SHAP value (log-odds)")

    style_axis(ax)

for ax in axes.ravel()[len(top_indices):]:
    ax.remove()

fig.suptitle(
    f"CatBoost SHAP Dependence - {set_label} Set",
    fontsize=17
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
save_show(fig, "CatBoost_SHAP_Dependence")


# ======================================================
# 9. 部分依赖图PDP
# 固定一个特征，对当前解释集的预测概率求平均
# 类别变量仅使用实际类别，不进行连续插值
# ======================================================

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5 * ncols, 4 * nrows),
    squeeze=False
)

pdp_results = {}

for ax, j in zip(axes.ravel(), top_indices):
    name = feature_names[j]
    is_categorical = name in cat_names

    column = X_explain[name].to_numpy()
    unique_values = np.sort(np.unique(column))

    if is_categorical or len(unique_values) <= 10:
        grid = unique_values
    else:
        low, high = np.quantile(column, [0.05, 0.95])
        grid = np.unique(np.linspace(low, high, 50))

    changed = X_explain.copy()
    average_prob = []

    for value in grid:
        # 按列赋值，保留其他类别列的整数类型
        changed[name] = (
            int(value) if is_categorical else float(value)
        )

        average_prob.append(
            source_model.predict_proba(changed)[:, 1].mean()
        )

    average_prob = np.asarray(average_prob)

    pdp_results[name] = pd.DataFrame({
        "Feature_value": grid,
        "Mean_predicted_probability": average_prob
    })

    if is_categorical:
        # 类别间不连线，避免暗示存在连续数量关系
        ax.scatter(
            grid,
            average_prob,
            color="#008F7A",
            s=65,
            zorder=3
        )
        ax.set_xticks(grid)
    else:
        ax.plot(
            grid,
            average_prob,
            color="#008F7A",
            linewidth=2.3,
            marker="o" if len(grid) <= 10 else None
        )

    ax.set_title(name, fontsize=13)
    ax.set_xlabel(name)
    ax.set_ylabel("Mean predicted probability")
    ax.set_ylim(0, 1)

    style_axis(ax)

for ax in axes.ravel()[len(top_indices):]:
    ax.remove()

fig.suptitle(
    f"CatBoost Partial Dependence - {set_label} Set",
    fontsize=17
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
save_show(fig, "CatBoost_Partial_Dependence_PDP")


# ======================================================
# 10. 决策图
# 每条线代表一个样本
# 从基准值累加SHAP贡献，终点为模型log-odds
# ======================================================

rng = np.random.default_rng(SEED)

decision_indices = np.sort(
    rng.choice(
        n_samples,
        size=min(N_DECISION, n_samples),
        replace=False
    )
)

fig = plt.figure(
    figsize=(10, max(6, 0.4 * n_features + 1.5))
)

shap.decision_plot(
    float(base_values[0]),
    shap_values[decision_indices],
    X_explain.iloc[decision_indices],
    feature_names=feature_names,
    feature_order=np.argsort(mean_abs_shap),
    feature_display_range=slice(None, None, -1),
    link="identity",
    alpha=0.6,
    auto_size_plot=False,
    show=False
)

ax = plt.gca()
ax.set_xlabel("Model output (log-odds)")
ax.set_title(
    f"CatBoost SHAP Decision - {set_label} Set "
    f"(n={len(decision_indices)})",
    pad=16
)

fig.tight_layout()
save_show(fig, "CatBoost_SHAP_Decision")


# ======================================================
# 11. 热力图
# 样本按预测概率升序排列
# 特征按平均绝对SHAP值降序排列
# ======================================================

plt.figure()

shap.plots.heatmap(
    shap_explanation,
    instance_order=np.argsort(pred_prob),
    feature_values=mean_abs_shap,
    feature_order=shap_feature_order,
    max_display=n_features,
    plot_width=12,
    show=False
)

fig = plt.gcf()

fig.suptitle(
    f"CatBoost SHAP Heatmap - {set_label} Set",
    y=1.03,
    fontsize=17
)

for ax in fig.axes:
    if ax.get_xlabel() == "Instances":
        ax.set_xlabel(
            "Samples ordered by predicted probability (low to high)"
        )
    if ax.get_ylabel().startswith("SHAP value"):
        ax.set_ylabel("SHAP value (log-odds)")

save_show(fig, "CatBoost_SHAP_Heatmap")

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from IPython.display import display


# ======================================================
# 1. 设置
# ======================================================

OUTPUT_DIR = Path(r"D:\两年内是否复发")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05

model_order = [
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "XGBoost",
    "CatBoost",
    "GBDT"
]

missing_models = [
    name for name in model_order
    if name not in best_models
]

if missing_models:
    raise ValueError(f"缺少模型：{missing_models}")

if not isinstance(X_test, pd.DataFrame):
    raise TypeError(
        "X_test必须保留DataFrame格式，请使用最新训练代码生成的数据。"
    )

y_true = np.asarray(y_test).ravel()

if len(X_test) != len(y_true):
    raise ValueError("测试集特征与标签的样本数不一致。")

if not np.array_equal(np.unique(y_true), [0, 1]):
    raise ValueError("测试集标签必须包含0和1两类。")

n_positive = int(np.sum(y_true == 1))
n_negative = int(np.sum(y_true == 0))

if min(n_positive, n_negative) < 2:
    raise ValueError("DeLong方差估计要求每个类别至少有2个样本。")


# ======================================================
# 2. 获取测试集预测得分
# 不使用分类阈值，不重新训练或校正模型
# ======================================================

test_scores = {}

for name in model_order:
    model = best_models[name]

    if name == "SVM":
        scores = model.decision_function(X_test)
    else:
        positive_index = np.flatnonzero(
            np.asarray(model.classes_) == 1
        )

        if positive_index.size != 1:
            raise ValueError(f"{name}的类别标签中未找到唯一正类1。")

        scores = model.predict_proba(X_test)[
            :, positive_index[0]
        ]

    scores = np.asarray(scores, dtype=float).ravel()

    if len(scores) != len(y_true):
        raise ValueError(f"{name}预测得分与标签长度不一致。")

    if not np.isfinite(scores).all():
        raise ValueError(f"{name}预测得分包含缺失值或无穷值。")

    test_scores[name] = scores


# ======================================================
# 3. 配对DeLong：AUC及协方差
# 正例分数 > 负例分数记1，相等记0.5，否则记0
# ======================================================

def delong_auc_covariance(y, score_matrix):
    y = np.asarray(y).ravel()
    score_matrix = np.asarray(score_matrix, dtype=float)

    positive_scores = score_matrix[:, y == 1]
    negative_scores = score_matrix[:, y == 0]

    m = positive_scores.shape[1]
    n = negative_scores.shape[1]

    positive_placements = []
    negative_placements = []
    auc_values = []

    for pos, neg in zip(positive_scores, negative_scores):
        comparison = (
            (pos[:, None] > neg[None, :]).astype(float)
            + 0.5 * (pos[:, None] == neg[None, :])
        )

        positive_placements.append(comparison.mean(axis=1))
        negative_placements.append(comparison.mean(axis=0))
        auc_values.append(comparison.mean())

    positive_placements = np.asarray(positive_placements)
    negative_placements = np.asarray(negative_placements)

    covariance = (
        np.atleast_2d(
            np.cov(positive_placements, rowvar=True, ddof=1)
        ) / m
        + np.atleast_2d(
            np.cov(negative_placements, rowvar=True, ddof=1)
        ) / n
    )

    return np.asarray(auc_values), covariance


score_matrix = np.vstack([
    test_scores[name] for name in model_order
])

auc_values, auc_covariance = delong_auc_covariance(
    y_true,
    score_matrix
)

# 验证计算的AUC与sklearn一致
np.testing.assert_allclose(
    auc_values,
    [
        roc_auc_score(y_true, test_scores[name])
        for name in model_order
    ],
    rtol=1e-10,
    atol=1e-10
)


# ======================================================
# 4. 各模型AUC及DeLong 95%CI
# 此处使用DeLong正态近似CI，与Bootstrap CI可能略有差异
# ======================================================

z_critical = norm.ppf(1 - ALPHA / 2)

auc_se = np.sqrt(
    np.maximum(np.diag(auc_covariance), 0)
)

auc_results = pd.DataFrame({
    "Model": model_order,
    "Test_AUC": auc_values,
    "SE_DeLong": auc_se,
    "CI_Lower": np.maximum(
        0, auc_values - z_critical * auc_se
    ),
    "CI_Upper": np.minimum(
        1, auc_values + z_critical * auc_se
    )
})


# ======================================================
# 5. 两两配对DeLong检验：双侧P值
# 差值定义为AUC(Model_1) - AUC(Model_2)
# ======================================================

comparison_rows = []

for i, j in combinations(range(len(model_order)), 2):
    difference = float(auc_values[i] - auc_values[j])

    variance = float(
        auc_covariance[i, i]
        + auc_covariance[j, j]
        - 2 * auc_covariance[i, j]
    )

    if variance < -1e-12:
        raise ValueError("AUC差值方差异常，请检查输入数据。")

    variance = max(variance, 0.0)
    se_difference = np.sqrt(variance)

    if se_difference > 1e-12:
        z_value = difference / se_difference
        p_value = 2 * norm.sf(abs(z_value))
        status = "OK"
    elif abs(difference) <= 1e-12:
        z_value = 0.0
        p_value = 1.0
        status = "Zero variance, equal AUC"
    else:
        # 方差为零但AUC不同，正态近似检验无法可靠计算
        z_value = np.nan
        p_value = np.nan
        status = "Zero variance, test undefined"

    comparison_rows.append({
        "Model_1": model_order[i],
        "Model_2": model_order[j],
        "AUC_1": auc_values[i],
        "AUC_2": auc_values[j],
        "AUC_Difference": difference,
        "Difference_CI_Lower": (
            difference - z_critical * se_difference
            if status != "Zero variance, test undefined"
            else np.nan
        ),
        "Difference_CI_Upper": (
            difference + z_critical * se_difference
            if status != "Zero variance, test undefined"
            else np.nan
        ),
        "Z": z_value,
        "P_DeLong": p_value,
        "Status": status
    })

delong_results = pd.DataFrame(comparison_rows)


# ======================================================
# 6. Holm多重比较校正：按全部15组比较校正
# ======================================================

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)

    # 无法计算的检验按1参与校正，最终仍保留NaN
    valid = np.isfinite(p_values)
    working_p = np.where(valid, p_values, 1.0)

    order = np.argsort(working_p)
    total = len(working_p)

    sorted_adjusted = np.maximum.accumulate(
        working_p[order] * (total - np.arange(total))
    )
    sorted_adjusted = np.minimum(sorted_adjusted, 1.0)

    adjusted = np.empty(total, dtype=float)
    adjusted[order] = sorted_adjusted
    adjusted[~valid] = np.nan

    return adjusted


delong_results["P_Holm"] = holm_adjust(
    delong_results["P_DeLong"].to_numpy()
)

delong_results["Significant_Holm"] = [
    "Not estimable" if not np.isfinite(p)
    else ("Yes" if p < ALPHA else "No")
    for p in delong_results["P_Holm"]
]


# ======================================================
# 7. P值矩阵及保存
# Excel保存完整精度，显示时再格式化
# ======================================================

p_raw_matrix = pd.DataFrame(
    np.eye(len(model_order)),
    index=model_order,
    columns=model_order
)

p_holm_matrix = p_raw_matrix.copy()

for _, row in delong_results.iterrows():
    name_1, name_2 = row["Model_1"], row["Model_2"]

    p_raw_matrix.loc[name_1, name_2] = row["P_DeLong"]
    p_raw_matrix.loc[name_2, name_1] = row["P_DeLong"]

    p_holm_matrix.loc[name_1, name_2] = row["P_Holm"]
    p_holm_matrix.loc[name_2, name_1] = row["P_Holm"]

output_file = OUTPUT_DIR / "测试集_六模型_AUC_DeLong检验.xlsx"

with pd.ExcelWriter(output_file) as writer:
    auc_results.to_excel(
        writer, sheet_name="Test_AUC", index=False
    )
    delong_results.to_excel(
        writer, sheet_name="Pairwise_DeLong", index=False
    )
    p_raw_matrix.to_excel(
        writer, sheet_name="P_Raw_Matrix"
    )
    p_holm_matrix.to_excel(
        writer, sheet_name="P_Holm_Matrix"
    )


# ======================================================
# 8. 仅显示两张结果表
# ======================================================

auc_display = auc_results[["Model"]].copy()

auc_display["Test AUC (DeLong 95% CI)"] = [
    f"{row.Test_AUC:.3f} "
    f"({row.CI_Lower:.3f}-{row.CI_Upper:.3f})"
    for row in auc_results.itertuples()
]

comparison_display = delong_results[
    ["Model_1", "Model_2"]
].copy()

comparison_display["AUC difference (95% CI)"] = [
    f"{row.AUC_Difference:.3f} "
    f"({row.Difference_CI_Lower:.3f}, "
    f"{row.Difference_CI_Upper:.3f})"
    for row in delong_results.itertuples()
]

def format_p(p):
    if not np.isfinite(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

comparison_display["P_DeLong"] = (
    delong_results["P_DeLong"].map(format_p)
)

comparison_display["P_Holm"] = (
    delong_results["P_Holm"].map(format_p)
)

comparison_display["Significant_Holm"] = (
    delong_results["Significant_Holm"]
)

display(auc_display)
display(comparison_display)